In [37]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [38]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [39]:
retriever = vectorstore.as_retriever()

In [40]:
from langchain.tools import tool

@tool
def search_documents(query: str) -> str:
    """2026 년 테크노빌드 주식회사(TechnoBuild) 임직원 
통합 가이드북입니다.
    """
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [41]:
system_prompt = """당신은 테크노빌드 주식회사 가이드북 정보를 친절하게 제공하는 어시스턴트입니다.

1. 정보가 필요할 경우 반드시 검색 도구(retriever_tool)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 억지로 꾸며내지 말고 모른다고 답변하세요.
"""

In [42]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents],
    system_prompt=system_prompt
)

In [43]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [44]:
from langchain.messages import SystemMessage, HumanMessage

res = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

In [45]:
print(res["messages"][-1].content[0]["text"])

테크노빌드 주식회사에서는 직무 관련 국가 기술 자격을 취득할 경우, 자격 등급에 따라 **1회성 축하금**과 **매월 지급되는 자격 수당**을 받으실 수 있습니다.

상세 지원 금액은 다음과 같습니다:

| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |
| :--- | :--- | :--- | :--- |
| **기술사 / 기능장** | 200만 원 | 30만 원 | 금속재료, 용접, 기계가공 등 |
| **기사** | 50만 원 | 10만 원 | 일반기계, 전기, 산업안전 등 |
| **산업기사** | 30만 원 | 5만 원 | 기계설계, 위험물 등 |
| **기능사** | 10만 원 | 3만 원 | 선반, 밀링, 특수용접 등 |

**주요 참고 사항:**
*   **지급 조건:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정됩니다. (더 상위 등급을 취득하면 수당이 갱신됩니다.)
*   **축하금:** 횟수 제한 없이 취득 시마다 지급됩니다.
*   **신청 및 지급:** 자격증 사본을 HR팀에 제출하면 축하금은 제출 후 2주 이내에 별도로 입금되며, 자격 수당은 제출한 다음 달 급여부터 반영됩니다.


---

In [46]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [47]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents],
    system_prompt=system_prompt,
)

In [48]:
query = "자격증 비용은 얼마를 받을 수 있어?"

In [49]:
from langchain.messages import SystemMessage, HumanMessage

res = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

In [50]:
res

{'messages': [HumanMessage(content='자격증 비용은 얼마를 받을 수 있어?', additional_kwargs={}, response_metadata={}, id='fa183261-dbd1-4a2b-8b4f-c39a8ecbeec7'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_documents', 'arguments': '{"query": "\\uc790\\uaca9\\uc99d \\uc9c0\\uc6d0 \\ube44\\uc6a9 \\ubc0f \\uc218\\ub2f9"}'}, '__gemini_function_call_thought_signatures__': {'17ad8908-b155-46cb-9ca5-b70271c895e5': 'Eu4ECusEAb4+9vuMUAaQyw58/+CBEnF90uD1gYY/YxoVY8C/SDi4FQf0R2Tj2ktdZIIyWQdAnH+pm0C9q9R+yuB0tlmOZYYUowettcuFv0FodUf5qBYdWWBNLvBs4AwwdtYVlWT6Ea2u8+2/YH8goiCAmytRccSeGuy1an+1I5J9IeMvsNESNjDQ7gHT4k+ssOxbzdcm+rOuwAeUyyDSU2c5GOiWqA5+Cusgd8xX4R6f99+pV37PMrPx6yq0wtiziPZE4dOJHXc/gGET9UCRoWNWK3QPoGxuCU/SDT6SaAL6O5bpkdoaxY/Mw9LokTsXAmUSRA2MjFPZjc/jjbfQqZ1fJa0+Pzx03y0LpDV1ocZwHUsw7zkfPF2RhQT7AH+8/m5/RjD/cgkQsvPm+QKdJplseGMsrMGNGsqWUCp+Vc4Pbmj2U23Z1Uh3EN1QfFKAbSysmb9afAN9uNZS6whtLZICQZiItiEaGLDSIGctYNnP6ejTssH2glRHnOEKLPXexi93HsA04hS7K3A7HADK2aw+t6UaIc7SuyoLNO1bJFoWLqNXSZKW4HclCZ

In [51]:
print(res["messages"][-1].content[0]["text"])

테크노빌드 주식회사에서는 직무 관련 국가 기술 자격 취득 시 **1회성 축하금**과 **매월 지급되는 자격 수당**을 받으실 수 있습니다. 자격 등급별 상세 금액은 다음과 같습니다.

### 1. 자격증 취득 축하금 (1회성)
*   **기술사/기능장:** 200만 원
*   **기사:** 50만 원
*   **산업기사:** 30만 원
*   **기능사:** 10만 원
    *   축하금은 횟수 제한 없이 지급됩니다.

### 2. 자격 수당 (월 지급)
*   **기술사/기능장:** 월 30만 원
*   **기사:** 월 10만 원
*   **산업기사:** 월 5만 원
*   **기능사:** 월 3만 원

### 주요 안내 사항
*   **지급 조건:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정되며, 상위 등급 취득 시 수당이 갱신됩니다.
*   **신청 및 지급 시기:** 
    *   자격증 사본을 HR팀에 제출한 **익월 급여부터** 자격 수당이 반영됩니다.
    *   축하금은 서류 제출 후 **2주 이내**에 별도로 입금됩니다.

해당하는 자격증(예: 일반기계, 전기, 산업안전 등)을 취득하신 후 HR팀에 사본을 제출하여 혜택을 받으시기 바랍니다.


---

In [ ]:
agent = create_agent(
        model="google_genai:gemini-3-flash-preview",
        # model="google_genai:gemini-2.5-flash-lite",
        tools=[search_documents],
        system_prompt=system_prompt,
    )

# 딕셔너리에서 마지막 AI 메시지 추출 (메시지가 없을 경우 대비)
def extract_ai_msg(data):
    if isinstance(data, dict) and "messages" in data:
        return data["messages"][-1]
    return AIMessage(content="")

# 메시지 객체에서 텍스트 내용만 추출 (Gemini의 리스트 구조 대응)
def extract_text(msg):
    if hasattr(msg, "content"):
        content = msg.content
        if isinstance(content, list) and len(content) > 0:
            if isinstance(content[0], dict) and "text" in content[0]:
                return content[0]["text"]
        return content
    return str(msg)

# 체인 구성: 에이전트 -> 메시지 추출 -> 텍스트 추출 -> 문자열 변환
chain = agent | extract_ai_msg | extract_text | StrOutputParser()

In [56]:
from langchain.messages import SystemMessage, HumanMessage

res = chain.invoke({
    "messages": [HumanMessage(content=query)]
})

In [57]:
res

"테크노빌드 주식회사에서는 직무 관련 국가 기술 자격을 취득할 경우, 자격 등급에 따라 **일시금인 '축하금'**과 **매월 지급되는 '자격 수당'**을 받으실 수 있습니다.\n\n상세 지원 금액은 다음과 같습니다:\n\n| 자격 등급 | 축하금 (1회성) | 자격 수당 (월) | 대상 자격증 예시 |\n| :--- | :--- | :--- | :--- |\n| **기술사 / 기능장** | 200만 원 | 30만 원 | 금속재료, 용접, 기계가공 등 |\n| **기사** | 50만 원 | 10만 원 | 일반기계, 전기, 산업안전 등 |\n| **산업기사** | 30만 원 | 5만 원 | 기계설계, 위험물 등 |\n| **기능사** | 10만 원 | 3만 원 | 선반, 밀링, 특수용접 등 |\n\n**[주요 조건 및 안내 사항]**\n*   **지급 조건:** 동일 등급 내에서는 1개의 자격증만 수당으로 인정되며, 상위 등급을 취득할 경우 수당이 갱신됩니다.\n*   **횟수 제한:** 축하금의 경우 취득 횟수에 제한 없이 지급됩니다.\n*   **지급 시기:** 축하금은 신청 서류 제출 후 2주 이내에 별도로 입금되며, 자격 수당은 급여 명세서에서 확인 가능합니다."